# Selection time, read time and tile granularity

Compare Paper greedy with tiles at the profiled reference width, then follow the tile family toward one-row Top-$R$. Coarser widths remain a second branch. Times, importance changes and rankings are computed from the saved session.

Run from this directory with NumPy, pandas, Matplotlib and Jupyter. Inputs are `measured_results.csv` and `measured_environment.json`; all interpretation and plotting code is below. The measurements use dense activation traces and a random file sized for weight rows. Retained activation importance is a proxy; model accuracy and full inference speed are outside this experiment.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

RESULTS, ENVIRONMENT = Path("measured_results.csv"), Path("measured_environment.json")
PAPER, TILE = "Paper greedy (GPU sort)", "Tiles (1s)"

## Quantities and timing

For $N$ rows and requested fraction $f$, $R=\max(1,\lfloor fN\rfloor)$. Importance $v_i$ is the mean absolute input activation over batch and token positions, normalized to mean one. For mask $M$,

$$q(M)=\frac{\sum_i v_iM_i}{\sum_i v_i},\qquad \operatorname{fill}(M)=\frac{|M|}{R}.$$

Both are reported as percentages. Zero-total importance leaves $q$ undefined. Paper accepts nonoverlapping windows ranked by importance per modeled cost and can underfill. Tiles select exactly $R$ rows using full tiles and a residual interval, comparing offsets zero and half a tile by importance per merged-run cost.

Selection starts with prepared CPU importance and includes Paper's GPU sorting and score transfers. Wall time ends when selected bytes reach CPU memory. Native I/O time covers the read/thread interval; wall time also includes read preparation. Adjacent rows merge and direct-I/O requests split at 768 KiB. Weight transfer to GPU and model computation are outside timing. At $R=N$, all methods bypass selection.

Methods are randomly interleaved within each trace and budget after warmup. Repetitions first collapse to case medians; each trace/budget case then has equal weight.

In [ ]:
frame = pd.read_csv(RESULTS)
assert not frame.empty, "Run 03_01_measure_selection_latency.ipynb before analyzing."
environment = json.loads(ENVIRONMENT.read_text())
display(pd.Series(environment)[["platform", "torch", "reader_threads", "warmup", "repeats", "traces",
                                "direct_io", "tile_boundary_kib", "paper_ceiling_kib", "mount"]])

In [ ]:
if "paper_tuning" in environment:
    display(pd.DataFrame({shape: {**t["best"], "feasible": t["num_feasible"]}
                          for shape, t in environment["paper_tuning"].items()}).T)
else:
    display(Markdown("Paper window tuning was not recorded in this saved session."))

## The recorded width grid

The profile boundary $s_{\rm KiB}$ minimizes read cost per KiB. With $w$ KiB per row, the geometric branch is $B_j=\max(1,\lceil(s_{\rm KiB}/w)/2^j\rceil)$, ending at one-row Top-$R$. The coarse branch spans $1s$ to $2s$. Widths are deduplicated within each shape; `tile_scale` is the requested multiplier and `tile_rows` the realized width.

The expected grid below uses the saved environment. Missing widths are reported explicitly, and any ranking covers only recorded widths. The reference boundary here comes from the measured profile; notebook 02 uses a fitted two-line boundary, so compare realized row widths when relating the two experiments.

In [ ]:
COARSE_SCALES = environment.get("coarse_scales", [1, 1.25, 1.5, 1.75, 2])
frame["method"] = frame.method.replace({"Saturation": TILE})
assert {"tile_rows", "tile_scale", "tile_kib", "intervals"} <= set(frame), "Record tile metadata first."
assert np.isfinite(frame.io_ms).all() and (frame.io_ms > 0).all()

In [ ]:
def tile_grid(s):
    levels = max(0, int(np.ceil(np.log2(max(1, s)))))
    scales = [(f"{a:g}s", float(a)) for a in COARSE_SCALES]
    scales += [(f"s/2^{j}", 2.**-j) for j in range(1, levels + 1)]
    seen, grid = set(), []
    for label, scale in scales:
        width = max(1, int(np.ceil(scale * s)))
        if width not in seen:
            name = "Top-R" if width == 1 else f"Tiles ({label})"
            grid.append((name, scale, width))
            seen.add(width)
    return grid

In [ ]:
base_methods, missing_widths = {}, {}
for shape, group in frame.groupby("shape"):
    w = int(shape.split("x")[1]) * environment["dtype_bytes"] / 1024
    s = environment["tile_boundary_kib"] / w
    actual = group[group.tile_rows.notna()][["method", "tile_rows"]].drop_duplicates()
    assert not actual.tile_rows.duplicated().any(), f"Duplicate tile widths in {shape}."
    baseline = actual[actual.tile_rows == max(1, int(np.ceil(s)))].method
    assert len(baseline) == 1, f"Reference width missing in {shape}."
    base_methods[shape] = baseline.iloc[0]
    missing_widths[shape] = sorted({B for _, _, B in tile_grid(s)} - set(actual.tile_rows))
for shape, missing in missing_widths.items():
    if missing:
        display(Markdown(f"**{shape}: incomplete sweep.** Unmeasured widths: {missing} rows. "
                         "Run the measurement notebook to acquire these points."))

## Case medians and paired methods

In [ ]:
values = ["selection_ms", "io_ms", "wall_ms", "retained_pct", "fill_pct"]
keys = ["shape", "trace_id", "fraction", "R_nominal"]
cases = frame.groupby(keys + ["method"])[values + ["R_selected", "tile_rows", "tile_kib"]].median().reset_index()
expected = cases.groupby("shape").method.agg(frozenset)
for key, methods in cases.groupby(keys).method.agg(frozenset).items():
    assert methods == expected.loc[key[0]] and PAPER in methods, "Unpaired methods within a shape."
is_base = cases.method.eq(cases["shape"].map(base_methods))
reference = cases[is_base | cases.method.eq(PAPER)]
display(reference.groupby(["shape", "method"])[values].median().round(4))

Each curve point is the median across traces at one requested fraction. The same points and axis ranges are reused in both comparisons. Lines connect increasing budgets; they do not infer latency at equal retained importance.

In [ ]:
all_points = cases.groupby(["shape", "method", "fraction"])[values].median().reset_index()
points = all_points[all_points.method.eq(PAPER) | all_points.method.eq(all_points["shape"].map(base_methods))]
metrics = [("io_ms", "I/O time (ms)"), ("selection_ms", "Selection time (ms)"), ("wall_ms", "Wall time (ms)")]
limits = all_points.groupby("shape")[[m for m, _ in metrics]].max().max(axis=1) * 1.05
names = cases.loc[cases.tile_rows.notna(), "method"].drop_duplicates().tolist()
colors = dict(zip(names, plt.get_cmap("viridis")(np.linspace(.05, .95, len(names)))))
colors.update({PAPER: "tab:blue", TILE: "black", "Top-R": "tab:red"})

In [ ]:
def plot_tradeoffs(points):
    for shape, group in points.groupby("shape"):
        fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), sharey=True)
        for ax, (metric, xlabel) in zip(axes, metrics):
            for name, data in group.groupby("method", sort=False):
                data = data.sort_values("fraction")
                ax.plot(data[metric], data.retained_pct, "o-", label=name, color=colors[name],
                        markersize=3, linewidth=2.2 if name == base_methods[shape] else 1)
            ax.set(xlabel=xlabel, xlim=(0, limits[shape]), ylim=(0, 105))
        axes[0].set_ylabel("Retained importance (%)")
        fig.suptitle(shape)
        handles, labels = axes[0].get_legend_handles_labels()
        fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(.5, .92), ncol=4, fontsize=8)
        fig.tight_layout(rect=(0, 0, 1, .72))
        plt.show()

In [ ]:
plot_tradeoffs(points)

## Reference tiles versus Paper

Subtract Paper from the reference tiles within each trace and requested budget, then take medians by shape. Negative time differences favor tiles; positive importance differences mean more retained importance, in percentage points. The dense control is included. Component medians are not added.

In [ ]:
paper = cases[cases.method == PAPER]
filled = int(paper.R_selected.eq(paper.R_nominal).sum())
display(Markdown(f"Paper fills **{filled:,}/{len(paper):,}** cases exactly; "
                 f"median fill is **{paper.fill_pct.median():.2f}%**."))
differences = cases[is_base].set_index(keys)[values] - paper.set_index(keys)[values]
display(differences.groupby("shape").median().rename(columns={
    "selection_ms": "Δ selection (ms)", "io_ms": "Δ I/O (ms)", "wall_ms": "Δ wall (ms)",
    "retained_pct": "Δ importance (pp)", "fill_pct": "Δ fill (pp)"}).round(4))

## Finer and coarser tiles

For each trace and budget, subtract the reference tile result before aggregating:

$$\Delta t_B=t_B-t_{\rm ref},\qquad \Delta q_B=100(q_B-q_{\rm ref}).$$

Negative time differences mean faster; positive importance differences mean more retained importance. Tables and rankings exclude $R=N$, where selection is bypassed. Curves retain that control. A smaller time alone does not establish a better importance–latency trade-off.

In [ ]:
scaled = cases[cases.tile_rows.notna() & (cases.fraction < 1)].copy()
assert not scaled.empty, "Record at least one budget below N."
baseline = scaled[scaled.method.eq(scaled["shape"].map(base_methods))].set_index(keys)[values]
scaled = scaled.join(baseline, on=keys, rsuffix="_ref", validate="many_to_one")
for metric in values:
    scaled[f"delta_{metric}"] = scaled[metric] - scaled[f"{metric}_ref"]
scale_metrics = ["tile_rows", "tile_kib", "wall_ms", "retained_pct",
                 "delta_selection_ms", "delta_io_ms", "delta_wall_ms", "delta_retained_pct"]
scale_summary = scaled.groupby(["shape", "method"])[scale_metrics].median()
display(scale_summary.round(4))

In [ ]:
plot_tradeoffs(all_points[all_points.method.isin(names)])

## Latency along the granularity axis

A base-two width axis gives equal spacing to successive halvings. These are medians over the recorded cases; no collapse threshold is assumed. Row widths make the geometric endpoint visible even when its requested multiplier differs across shapes.

In [ ]:
for shape, group in scale_summary.groupby(level="shape"):
    group = group.sort_values("tile_rows")
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.3))
    for ax, metric, label in zip(axes, ["wall_ms", "retained_pct"], ["Wall time (ms)", "Retained importance (%)"]):
        ax.plot(group.tile_rows, group[metric], "o-")
        ax.set_xscale("log", base=2)
        ax.invert_xaxis()
        ax.set(xlabel="Tile width (rows)", ylabel=label)
    fig.suptitle(shape)
    fig.tight_layout()
    plt.show()

## Selected channels at one budget

Reconstruct masks from recorded intervals. By default, use the lowest trace ID and the measured fraction nearest 0.5, favoring the smaller fraction in a tie. Set the two variables below to inspect another case. This illustrates contiguity for one case.

In [ ]:
MASK_TRACE_ID, MASK_FRACTION = None, 0.5
pool = frame[frame.tile_rows.notna() & (frame.fraction < 1)]
trace_id = pool.trace_id.min() if MASK_TRACE_ID is None else MASK_TRACE_ID
sample = pool[pool.trace_id == trace_id]
assert not sample.empty, "Choose a recorded trace ID."
fraction = min(sample.fraction.unique(), key=lambda f: (abs(f-MASK_FRACTION), f))
sample = sample[sample.fraction == fraction]
assert sample.groupby("method").intervals.nunique().eq(1).all(), "Masks differ across repetitions."
example = sample.drop_duplicates("method").sort_values("tile_rows", ascending=False).set_index("method")
assert example[["shape", "R_nominal"]].nunique().eq(1).all()
shape, R = example["shape"].iloc[0], int(example.R_nominal.iloc[0])
N = int(shape.split("x")[0])

In [ ]:
masks = np.zeros((len(example), N), dtype=np.uint8)
labels = []
for row, (name, data) in enumerate(example.iterrows()):
    spans = json.loads(data.intervals)
    for start, end in spans:
        assert 0 <= start < end <= N
        masks[row, start:end] = 1
    assert int(masks[row].sum()) == int(data.R_selected) == R
    labels.append(f"{name}: B={int(data.tile_rows)}, selected={R}, runs={len(spans)}")

Yellow marks selected rows and purple marks omitted rows. The reference width is labeled in bold.

In [ ]:
fig, ax = plt.subplots(figsize=(14, max(3, len(example)*.35)))
ax.imshow(masks, aspect="auto", interpolation="nearest", cmap="viridis", vmin=0, vmax=1)
ax.set(yticks=np.arange(len(labels)), yticklabels=labels,
       xticks=np.unique(np.linspace(0, N-1, min(8, N), dtype=int)),
       xlabel=f"Channel (trace {trace_id}; R={R}; fraction={fraction:g})", title=shape)
ax.get_yticklabels()[list(example.index).index(base_methods[shape])].set_fontweight("bold")
fig.tight_layout()
plt.show()

## Reading the measured sweep

Report all recorded widths tied for the lowest median wall time, together with their paired importance changes. Missing widths remain unmeasured; the ranking covers only the recorded session. Near ties can change between runs, and the profile boundary need not minimize total time after selection, merging and threaded reads.

In [ ]:
findings = []
for shape, group in scale_summary.groupby(level="shape"):
    group = group.droplevel("shape")
    for name, row in group[group.wall_ms == group.wall_ms.min()].iterrows():
        quality = f"{row.delta_retained_pct:+.3f} pp" if pd.notna(row.delta_retained_pct) else "undefined"
        findings.append(f"- **{shape}**: among recorded widths, **{name} ({int(row.tile_rows)} rows)** "
                        f"has lowest median wall time **{row.wall_ms:.4f} ms**. Paired changes: "
                        f"**{row.delta_wall_ms:+.4f} ms**, **{quality}** importance.")
display(Markdown("\n".join(findings)))